In [191]:
import pandas as pd
from pathlib import Path
import numpy as np
import sys
sys.path.append("../src")
from cleaning import choose_pollutant_danger

DF_PATH = Path("../data/processed/openaq_cleaned.parquet")
df = pd.read_parquet(DF_PATH)

In [192]:
# look at the df
df.head()

,Country Code,City,Location,Coordinates,Pollutant,Source Name,Unit,Last Updated,Country Label,Value
0,BE,Unknown,Escautpont,"50.420270857658636, 3.551812869268811",SO2,EEA France,µg/m³,2017-07-18 20:00:00+00:00,Belgium,3.60000
1,BG,Teleorman-RNMCA,NET-RO058A,"43.650721999999995, 25.363583",CO,EEA Romania,µg/m³,2024-03-11 08:00:00+00:00,Bulgaria,1237.25114
2,BG,National air network,NET-BG001A,"42.518891999999994, 27.375144",O3,EEA Bulgaria,µg/m³,2024-03-11 07:00:00+00:00,Bulgaria,12.06000
3,BG,National air network,NET-BG001A,"42.669796999999996, 23.268403000000003",NO,EEA Bulgaria,µg/m³,2024-03-11 07:00:00+00:00,Bulgaria,42.43000
4,BG,National air network,NET-BG001A,"43.217279999999995, 27.935959999999998",NO2,EEA Bulgaria,µg/m³,2024-03-11 07:00:00+00:00,Bulgaria,16.60000


In [ ]:
# to predict Value, we should know yesterday's pollution value
# first we sort by date
df = df.sort_values(by=["Location", "Pollutant", "Last Updated"])
# create the feature showing what the value was 1 day ago
df["Val Lag 1 Day"] = df.groupby(["Location", "Pollutant"])["Value"].shift(1).astype("float64")

In [194]:
# create datetime-related features
df["Year"] = df["Last Updated"].dt.year.astype("int16")
df["Month"] = df["Last Updated"].dt.month.astype("int8")
df["Day"] = df["Last Updated"].dt.day.astype("int8")

# drop Last Updated Now
df.drop("Last Updated", axis=1, inplace=True)

In [195]:
# transform Month and Day into sine and cosine features for machine learning
df["Month Sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month Cos"] = np.cos(2 * np.pi * df["Month"] / 12)

In [196]:
# split Coordinates into latitude and longitude
df[["Latitude", "Longitude"]] = df.Coordinates.str.split(",", expand=True, regex=False).astype("float64").round(decimals=4)

# drop Coordinates now
df.drop("Coordinates", axis=1, inplace=True)

In [197]:
# create Has Coordinates feature to see whether the measure has coordinates or not
# remember that there were some NaN values in Coordinates
df["Has Coordinates"] = ((df.Latitude.notna()) & (df.Longitude.notna())).astype("int8")

In [198]:
# is City unknown?
df["Has City"] = (df.City != "Unknown").astype("int8")

In [199]:
# create a danger category based on the pollutant type
df["Pollutant Danger"] = df.Pollutant.map(choose_pollutant_danger).astype("category")

In [200]:
# move target to the end
col_to_move = df.pop("Value")
df.insert(len(df.columns), "Value", col_to_move)

In [201]:
# shuffle the df
df = df.sample(frac=1).reset_index(drop=True)

In [202]:
# look at the final data types
df.dtypes

Country Code        category
City                     str
Location                 str
Pollutant           category
Source Name              str
Unit                category
Country Label       category
Val Lag 1 Day        float64
Year                   int16
Month                   int8
Day                     int8
Month Sin            float64
Month Cos            float64
Latitude             float64
Longitude            float64
Has Coordinates         int8
Has City                int8
Pollutant Danger    category
Value                float64
dtype: object

In [203]:
# look at the final df
df.head()

,Country Code,City,Location,Pollutant,Source Name,Unit,Country Label,Val Lag 1 Day,Year,Month,Day,Month Sin,Month Cos,Latitude,Longitude,Has Coordinates,Has City,Pollutant Danger,Value
0,CN,Unknown,十里堡居委,PM2.5,ChinaAQIData,µg/m³,China,NaN,2021,8,9,-0.866025,-5.000000e-01,35.0215,118.3564,1,0,Most Dangerous,39.000000
1,ZA,City of Cape Town,Cape Point-NAQI,NO,South Africa,ppm,South Africa,NaN,2023,5,30,0.500000,-8.660254e-01,-34.3533,18.4898,1,1,Highly Harmful,0.000537
2,IT,RETE REGIONALE UMBRIA,NET.IT252A,SO2,EEA Italy,µg/m³,Italy,1.3,2024,3,9,1.000000,6.123234e-17,43.1031,12.3661,1,1,Highly Harmful,1.300000
3,CN,Unknown,太平,PM10,ChinaAQIData,µg/m³,China,NaN,2021,8,9,-0.866025,-5.000000e-01,41.1442,123.0485,1,0,Moderate,41.000000
4,US,Houston,Galveston 99th St. C1034/A320/X183,O3,Texas,ppm,United States,NaN,2016,3,6,1.000000,6.123234e-17,29.2545,-94.8613,1,1,Highly Harmful,0.062000


In [204]:
# save the feature engineered df
df.to_parquet("../data/processed/openaq_feature_engineered.parquet", index=False)